In [1]:
if __name__ != "__main__":
    raise Exception("This file is not meant to be imported")

In [ ]:
import sacrebleu
import json
from pathlib import Path
from pythainlp.tokenize import word_tokenize
from tqdm import tqdm

In [ ]:
fp_reference = Path("lectrans-testset/25.11.slides_annot/slide_translation_annot_by_sbj_all_text.json")
assert fp_reference.exists(), f"File not found: {fp_reference}"

In [ ]:
with open(fp_reference, 'r', encoding="utf8") as f:
    golds = json.load(f)

In [ ]:
def correcting_lang_id(lang):
    return {
        "jp": "ja"
    }.get(lang, lang)

In [ ]:
def tokenize_thai(text):
    return " ".join(word_tokenize(text, engine="newmm"))

In [ ]:
# import MeCab
# from rouge_score import rouge_scorer, tokenizer

# scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# class MecabTokenizer(tokenizer.Tokenizer):
#     def __init__(self, dic_path=None):
#         self.tagger = MeCab.Tagger('-Owakati')

#     def tokenize(self, text):
#         return self.tagger.parse(text).split()

# scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], tokenizer=MecabTokenizer())
# scores = scorer.score(ref, pred)


In [ ]:
def get_scores(golds, preds):
    scores_all = {}
    for sbj_lecid_pair, cur_gold in golds.items():
        sbj, lec_id = sbj_lecid_pair.split('/', 1)

        scores_per_subject = {}
        for scene_id, gold_all_langs in cur_gold.items():
            scores_per_scene = {}
            for lang, all_bb in gold_all_langs.items():
                lang = correcting_lang_id(lang)
                if lang == 'en':
                    continue
                gold = ['\n'.join(bb_text_list) for bb_text_list in all_bb.values()]
                pred = ['\n'.join(bb_text_list) for bb_text_list in preds[sbj_lecid_pair][scene_id][lang].values()]
                kwargs = {}
                match lang:
                    case "ja":
                        kwargs["tokenize"] = "ja-mecab"
                    case "zh":
                        kwargs["tokenize"] = "zh"
                    case "th":
                        kwargs["tokenize"] = "none"
                        gold = list(map(tokenize_thai, gold))
                        pred = list(map(tokenize_thai, pred))
                scores_per_scene[lang] = (
                    sacrebleu.corpus_bleu(pred, [gold], **kwargs).score,
                    sacrebleu.corpus_chrf(pred, [gold]).score,
                    # [scorer.score(g, p).fmeasure for g, p in zip(gold, pred)],
                )
            scores_per_subject[scene_id] = scores_per_scene
        scores_all[sbj_lecid_pair] = scores_per_subject
    return scores_all

In [ ]:
%%time
fd_outs_basedir = Path("outs_slidetextonly")
fd_evals_basedir = Path("evals_slides")
for fd_preds in tqdm(sorted(fd_outs_basedir.iterdir())):
    if not fd_preds.is_dir():
        continue
    preds = {}
    for fp in sorted(fd_preds.glob("*.json")):
        with open(fp, 'r', encoding="utf8") as f:
            cur_pred = json.load(f)
        assert len(cur_pred) == 1
        assert list(cur_pred.keys())[0] not in preds

        # Convert GPT dict_keys due to previous mishandling during query
        _key = sorted(cur_pred.keys())[0]
        _cur_pred = {}
        for _key_pred in cur_pred[_key].keys():
            for _key_gold in golds[_key].keys():
                if _key_gold.endswith(_key_pred):
                    break
            _cur_pred[_key_gold] = cur_pred[_key][_key_pred]
        preds[_key] = _cur_pred

    scores = get_scores(golds, preds)
    fd_evals_basedir.mkdir(exist_ok=True)
    with open(fd_evals_basedir / f"{fd_preds.name}_nocontext.json", 'w', encoding="utf8") as f:
        json.dump(scores, f, ensure_ascii=False, indent=2)

In [ ]:
fd_outs_basedir = Path("outs_slides")
fd_evals_basedir = Path("evals_slides")
for fd_preds in tqdm(sorted(fd_outs_basedir.iterdir())):
    if not fd_preds.is_dir():
        continue
    preds = {}
    for k, v in golds.items():
        sbj, lec_id = k.split('/', 1)
        fp_preds = sorted(fd_preds.glob("*.json"))
        for fp in fp_preds:
            if fp.name.startswith(f"{sbj}_{lec_id}"):
                assert k not in preds
                with open(fp, 'r', encoding="utf8") as f:
                    preds[k] = json.load(f)
        # True
        # assert all(
        #     sum(
        #         fp.name.startswith('_'.join(sbj_lecid_pair.split('/', 1)))
        #         for fp in fp_preds
        #     ) == 1
        #     for sbj_lecid_pair in golds.keys()
        # )

    scores = get_scores(golds, preds)
    fd_evals_basedir.mkdir(exist_ok=True)
    with open(fd_evals_basedir / f"{fd_preds.name}.json", 'w', encoding="utf8") as f:
        json.dump(scores, f, ensure_ascii=False, indent=2)